# Control Actions Analysis — 03LIC_1071 Filtered Episodes

Analysis of the most operated OP and SP tags across the 100 filtered alarm episodes for `03LIC_1071`.

**Data source**: `RESULTS/03LIC_1071_episodes_02JUN2026_1426/03LIC_1071_filtered_episodes/`

Each episode folder contains:
- `episode_XXXX_events.csv` — CHANGE events (control actions) for that episode window
- `episode_XXXX_pv_data.csv` — Minute-wise PV/OP data for that episode window

**Phases** are determined by comparing each event timestamp to the episode's `alarm_start` and `alarm_end`:
- **Before**: event timestamp < alarm_start  
- **During**: alarm_start ≤ event timestamp ≤ alarm_end  
- **After**: event timestamp > alarm_end

In [5]:
import re
import glob
import pandas as pd
from IPython.display import display, Markdown

# ── Paths ──────────────────────────────────────────────────────────────────────
EPISODES_BASE = (
    "../RESULTS/03LIC_1071_episodes_02JUN2026_1426"
    "/03LIC_1071_filtered_episodes"
)
XLSX_PATH = (
    "../RESULTS/03LIC_1071_episodes_02JUN2026_1426"
    "/03LIC_1071_pvlo_alarms_clustered_with_control_actions.xlsx"
)

# ── Load alarm episode metadata ────────────────────────────────────────────────
# Episode folder names correspond to cluster_id values (not episode_num).
# Each cluster may contain multiple alarm sub-episodes; we use
# cluster_start_time (first alarm start) and cluster_end_time (last alarm end)
# as the "during" window boundaries.
alarm_meta = pd.read_excel(XLSX_PATH, sheet_name="alarm_clusters")
alarm_meta["cluster_start_time"] = pd.to_datetime(alarm_meta["cluster_start_time"])
alarm_meta["cluster_end_time"]   = pd.to_datetime(alarm_meta["cluster_end_time"])

# One row per cluster: min cluster_start_time and max cluster_end_time
cluster_bounds = (
    alarm_meta.groupby("cluster_id")
    .agg(alarm_start=("cluster_start_time", "min"),
         alarm_end=("cluster_end_time",   "max"))
)

print(f"Alarm metadata loaded: {len(alarm_meta)} episodes across {len(cluster_bounds)} clusters")
print(cluster_bounds.head(3).to_string())

Alarm metadata loaded: 1379 episodes across 539 clusters
                       alarm_start               alarm_end
cluster_id                                                
1          2022-01-05 08:53:41.853 2022-01-05 09:33:33.105
2          2022-01-07 09:55:16.253 2022-01-07 10:00:27.504
3          2022-01-07 13:33:25.702 2022-01-07 13:36:35.204


In [6]:
# ── Load all events files and combine into one DataFrame ──────────────────────
# Episode folder names use cluster_id as the identifier.
event_files = glob.glob(f"{EPISODES_BASE}/**/episode_*_events.csv", recursive=True)
event_files = sorted(event_files)

dfs = []
for fpath in event_files:
    cluster_id = int(re.search(r"episode_(\d+)_events", fpath).group(1))
    df = pd.read_csv(fpath, low_memory=False)
    df["cluster_id"] = cluster_id
    dfs.append(df)

all_events = pd.concat(dfs, ignore_index=True)
all_events["VT_Start"] = pd.to_datetime(all_events["VT_Start"])

print(f"Total events loaded : {len(all_events):,}  across {len(event_files)} episodes")
print(f"Condition name counts:\n{all_events['ConditionName'].value_counts().to_string()}")

Total events loaded : 2,735  across 100 episodes
Condition name counts:
ConditionName
CHANGE    2735


In [7]:
# ── Filter to control actions: CHANGE events on OP or SP tags ─────────────────
control_actions = all_events[
    (all_events["ConditionName"] == "CHANGE") &
    (all_events["Description"].isin(["OP", "SP"]))
].copy()

# ── Assign phase (before / during / after) using cluster_start_time & cluster_end_time ─
def assign_phase(row):
    cid = row["cluster_id"]
    if cid not in cluster_bounds.index:
        return "unknown"
    a_start = cluster_bounds.loc[cid, "alarm_start"]
    a_end   = cluster_bounds.loc[cid, "alarm_end"]
    t = row["VT_Start"]
    if t < a_start:
        return "before"
    elif t <= a_end:
        return "during"
    else:
        return "after"

control_actions["phase"] = control_actions.apply(assign_phase, axis=1)

print(f"Total control actions: {len(control_actions):,}")
print(f"\nBy action type (Description):\n{control_actions['Description'].value_counts().to_string()}")
print(f"\nBy phase:\n{control_actions['phase'].value_counts().to_string()}")

Total control actions: 2,508

By action type (Description):
Description
OP    1780
SP     728

By phase:
phase
before    1685
during     440
after      383


---
## Overall: Most Operated OP and SP Tags
Count of control action events per tag across **all episodes** and **all phases**.

In [4]:
import pandas as pd

df = pd.read_parquet('/home/h604827/ControlActions/DATA/combined_events/03LIC_1071_PVLO_PVHI_combined_events.parquet')
df[['Source', 'SourceParameter', 'EventID']]

,Source,SourceParameter,EventID
0,SI_2K101_BN_LCHK,None,31631368
1,SI_2K101_BN_LCHK,None,31631545
2,03PI_3281,None,31630787
3,03PI_3281,None,31630786
4,None,None,31630786
...,...,...,...
5573152,03FIC_3227,None,79474266
5573153,03FIC_3227,None,79474266
5573154,None,None,79474267
5573155,None,None,79474266


In [2]:
df.columns

Index(['Action', 'Actor', 'AreaName', 'AlarmLimit', 'Block', 'Category',
       'ConditionName', 'Description', 'EventID', 'Flags', 'LocalTime',
       'LocationFullName', 'LocationTagName', 'PrevValue', 'Priority',
       'ReceivedDelay', 'ServerName', 'ShelvedReason', 'Source',
       'SourceParameter', 'Station', 'Time', 'TransactionID', 'Units', 'Value',
       'VT_Start', 'H', 'TagID', 'AlarmStatus', 'S', 'source_table',
       'IntervalIdentifier', 'Parameter', 'FromValue', 'ToValue', 'ACTION',
       'Unit', 'UnitOfMeasure', 'TagName', 'Area', 'Limit', 'EventTypeID',
       'VT_End', 'LastEvent', 'NextEvent', 'rowid', 'Seconds', 'PrevSeconds'],
      dtype='object')

In [7]:
df['source_table'].value_counts()

source_table
1A_ChangeEvents         2507555
1L_ChangeEvents         1051748
1K_ChangeEvents          523768
1H_ChangeEvents          277370
1A_AlarmEventsLinked     201117
1E_ChangeEvents          142167
1F_ChangeEvents          138249
1I_ChangeEvents           91578
1L_AlarmEvents            86427
1H_AlarmEvents            83813
1K_AlarmEvents            82657
1E_AlarmEvents            72000
1A_AlarmEvents            54844
1I_AlarmEvents            54842
1L_AlarmEventsLinked      49369
1F_AlarmEvents            44033
1E_AlarmEventsLinked      34706
1H_AlarmEventsLinked      32728
1K_AlarmEventsLinked      19819
1F_AlarmEventsLinked      19238
1I_AlarmEventsLinked       5129
Name: count, dtype: int64

In [9]:
def make_top_tags_table(df, action_type, top_n=20):
    """Return a ranked DataFrame of most operated tags for a given action type (OP or SP)."""
    subset = df[df["Description"] == action_type]
    counts = (
        subset.groupby("Source")
        .agg(
            num_actions=("Source", "count"),
            num_episodes=("cluster_id", "nunique"),
        )
        .sort_values("num_actions", ascending=False)
        .head(top_n)
        .reset_index()
    )
    counts.index = counts.index + 1          # 1-based rank
    counts.index.name = "rank"
    counts.columns = ["tag", "num_actions", "num_episodes"]
    return counts


# Overall OP tags
overall_op = make_top_tags_table(control_actions, "OP")
# Overall SP tags
overall_sp = make_top_tags_table(control_actions, "SP")

display(Markdown("### Overall — Most Operated **OP** Tags"))
display(overall_op.style.set_table_styles([
    {"selector": "th", "props": [("text-align", "center")]},
    {"selector": "td", "props": [("text-align", "center")]},
]).background_gradient(subset=["num_actions"], cmap="Blues"))

display(Markdown("### Overall — Most Operated **SP** Tags"))
display(overall_sp.style.set_table_styles([
    {"selector": "th", "props": [("text-align", "center")]},
    {"selector": "td", "props": [("text-align", "center")]},
]).background_gradient(subset=["num_actions"], cmap="Greens"))

### Overall — Most Operated **OP** Tags

,tag,num_actions,num_episodes
rank,,,
1,03PIC_1013,702,30
2,03HIC_1151,201,11
3,03LIC_1071,139,9
4,03FIC_3435,113,28
5,03HIC_3100,96,6
6,03HIC_1092A,76,1
7,03FIC_3415,61,4
8,03LIC_3408,50,1
9,03FIC_1085,35,2


### Overall — Most Operated **SP** Tags

,tag,num_actions,num_episodes
rank,,,
1,03LIC_1034,445,44
2,03TIC_1009,122,24
3,03LIC_1085,67,13
4,03LIC_1071,34,13
5,03LIC_1016,25,7
6,03LIC_3178,8,3
7,03LIC_3408,8,2
8,03LIC_3153,7,3
9,03LIC_1097,5,3


---
## Phase-wise Analysis

For each phase, count how many times each tag was operated (OP or SP CHANGE events).

- **Before** — events in the episode window *before* the alarm started  
- **During** — events *while the alarm was active* (alarm_start → alarm_end)  
- **After** — events in the episode window *after* the alarm cleared

In [10]:
PHASE_COLORS = {
    "before": "Oranges",
    "during": "Reds",
    "after":  "Purples",
}

for phase in ["before", "during", "after"]:
    phase_df = control_actions[control_actions["phase"] == phase]
    total = len(phase_df)
    display(Markdown(f"---\n### Phase: **{phase.upper()}**  ({total:,} total actions)"))

    for action_type, cmap in [("OP", "Blues"), ("SP", "Greens")]:
        tbl = make_top_tags_table(phase_df, action_type)
        if tbl.empty:
            display(Markdown(f"_No **{action_type}** actions in the **{phase}** phase._"))
        else:
            display(Markdown(f"**Most operated {action_type} tags — {phase}**"))
            display(
                tbl.style.set_table_styles([
                    {"selector": "th", "props": [("text-align", "center")]},
                    {"selector": "td", "props": [("text-align", "center")]},
                ]).background_gradient(subset=["num_actions"], cmap=cmap)
            )

---
### Phase: **BEFORE**  (1,685 total actions)

**Most operated OP tags — before**

,tag,num_actions,num_episodes
rank,,,
1,03PIC_1013,491,27
2,03FIC_3435,92,22
3,03HIC_3100,89,5
4,03HIC_1151,81,10
5,03HIC_1092A,75,1
6,03FIC_3415,52,3
7,03LIC_1071,52,5
8,03LIC_3408,50,1
9,03HIC_1009B,34,1


**Most operated SP tags — before**

,tag,num_actions,num_episodes
rank,,,
1,03LIC_1034,280,34
2,03TIC_1009,72,18
3,03LIC_1085,43,8
4,03LIC_1016,11,3
5,03LIC_3408,8,2
6,03LIC_3178,7,3
7,03LIC_3153,6,3
8,03LIC_1097,5,3
9,03LIC_1071,4,2


...

---
### Phase: **AFTER**  (383 total actions)

**Most operated OP tags — after**

,tag,num_actions,num_episodes
rank,,,
1,03PIC_1013,61,13
2,03HIC_1151,47,6
3,03LIC_1071,34,5
4,03LIC_1094,20,1
5,03FIC_3435,11,9
6,03FIC_3415,7,1
7,03HIC_3100,5,1
8,03HIC_1183,3,2
9,03LI_1035_MOS,2,1


**Most operated SP tags — after**

,tag,num_actions,num_episodes
rank,,,
1,03LIC_1034,88,14
2,03TIC_1009,27,4
3,03LIC_1085,19,5
4,03LIC_1071,10,5
5,03LIC_1016,8,1
6,03PIC_3131,5,1
7,03LIC_3178,1,1
8,03LIC_3153,1,1


In [22]:
# Episodes where 03LIC_1071 OP actions were taken
lic1071_op = control_actions[
    (control_actions["Source"] == "03LIC_1071") &
    (control_actions["Description"] == "OP")
][["cluster_id", "VT_Start", "PrevValue", "Value", "phase"]].copy()

lic1071_op_by_episode = (
    lic1071_op.groupby("cluster_id")
    .agg(
        num_actions=("cluster_id", "count"),
        phases=("phase", lambda x: ", ".join(sorted(x.unique()))),
        first_action_time=("VT_Start", "min"),
        last_action_time=("VT_Start", "max"),
    )
    .sort_values("num_actions", ascending=False)
    .reset_index()
)
lic1071_op_by_episode.index = lic1071_op_by_episode.index + 1
lic1071_op_by_episode.index.name = "rank"

display(Markdown("### Episodes with `03LIC_1071` OP Actions"))
display(lic1071_op_by_episode.style.set_table_styles([
    {"selector": "th", "props": [("text-align", "center")]},
    {"selector": "td", "props": [("text-align", "left")]},
]).background_gradient(subset=["num_actions"], cmap="Blues"))

### Episodes with `03LIC_1071` OP Actions

,cluster_id,num_actions,phases,first_action_time,last_action_time
rank,,,,,
1,498,36,during,2025-05-07 23:41:04.337200,2025-05-08 01:01:58.231600
2,250,26,"before, during",2024-01-18 09:52:51.056900,2024-01-18 09:59:22.855600
3,485,21,"after, before",2025-01-16 05:32:40.585000,2025-01-16 10:05:39.769500
4,434,16,"before, during",2024-09-26 08:05:41.145700,2024-09-26 09:22:32.893200
5,480,14,after,2025-01-12 07:01:49.908900,2025-01-12 07:34:06.732300
6,471,8,"after, during",2025-01-08 03:00:44.933000,2025-01-08 04:00:05.221900
7,441,7,before,2024-10-16 09:14:31.753100,2024-10-16 09:19:12.956600
8,470,6,after,2025-01-07 03:01:54.126300,2025-01-07 03:06:07.090100
9,499,5,"after, before, during",2025-05-08 05:38:05.790000,2025-05-08 06:42:53.857700
